# ⚡ AgroIntelli — Notebook 02: High-Precision MobileNetV3 Multimodal Retraining
### Complete 39-Class Model Training (PlantVillage + PlantDoc Fused Dataset)

## 🎯 Key Optimization Strategy
1. **Infinite `.repeat()` tf.data Stream**: Prevents generator exhaustion and fixes the `UserWarning: Your input ran out of data` error.
2. **MobileNetV3 Preprocessing**: Uses official `mobilenet_v3.preprocess_input` for exact feature alignment.
3. **Data Augmentation Layer**: In-graph RandomFlip, Rotation, Zoom, and Contrast jitter for outdoor field robustness.
4. **Label Smoothing Cross-Entropy**: Categorical cross-entropy with `label_smoothing=0.1` to prevent overconfidence.
5. **Two-Stage Fine-Tuning**: Stage 1 (Frozen backbone, 5 epochs) → Stage 2 (Fine-tuning top 35 layers, 15 epochs) with LR reduction and EarlyStopping.
6. **Failsafe Model Safeguard**: Automatically creates `agrointelli_phase1_final.backup.keras` before training and restores it if interrupted.

In [1]:
import os
import sys
import gc
import json
import shutil
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split

# RAM & Execution Safety
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
WEATHER_DIM = 4
BATCH_SIZE = 16

# Load canonical 39-class map
CLASS_MAP_PATH = Path("../backend/models/class_index_map.json")
if CLASS_MAP_PATH.exists():
    with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
        CLASS_TO_IDX = json.load(f)
    CLASS_NAMES = [k for k, v in sorted(CLASS_TO_IDX.items(), key=lambda x: x[1])]
    NUM_CLASSES = len(CLASS_NAMES)
else:
    print("⚠️ class_index_map.json not found. Run Notebook 01 first!")
    NUM_CLASSES = 39
    CLASS_NAMES = [f"class_{i}" for i in range(NUM_CLASSES)]
    CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

print(f"✅ Loaded {NUM_CLASSES} Unified Target Classes.")

✅ Loaded 39 Unified Target Classes.


## 📦 1. Discover Fused Dataset & Infinite tf.data Generator Stream

In [2]:
RAW_DIR = Path("./agrointelli_data/data/raw")

def discover_fused_dataset(raw_dir):
    records = []
    for root, _, files in os.walk(raw_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = Path(root) / file
                folder_name = full_path.parent.name.lower().replace("___", "_").replace(" ", "_")
                matched_cls = None
                for cls in CLASS_NAMES:
                    tokens = cls.split("_")
                    if all(t in folder_name for t in tokens if len(t) > 2):
                        matched_cls = cls
                        break
                if matched_cls:
                    source = "PlantDoc" if "plantdoc" in str(full_path).lower() else "PlantVillage"
                    records.append({
                        "file_path": str(full_path),
                        "class_name": matched_cls,
                        "class_idx": CLASS_TO_IDX[matched_cls],
                        "source": source
                    })
    return pd.DataFrame(records)

df_dataset = discover_fused_dataset(RAW_DIR)
print(f"✅ Discovered {len(df_dataset)} Total Images across PlantVillage + PlantDoc.")

def synthesize_weather_vector(class_name):
    cn = str(class_name).lower()
    if "late_blight" in cn or "mold" in cn:
        temp = np.random.normal(19.0, 3.0)
        humidity = np.random.normal(88.0, 5.0)
        rain = np.random.exponential(3.5)
        wind = np.random.normal(12.0, 4.0)
    elif "early_blight" in cn or "spot" in cn or "scab" in cn:
        temp = np.random.normal(26.0, 3.5)
        humidity = np.random.normal(78.0, 6.0)
        rain = np.random.exponential(1.5)
        wind = np.random.normal(14.0, 5.0)
    elif "rust" in cn or "powdery" in cn:
        temp = np.random.normal(22.0, 4.0)
        humidity = np.random.normal(72.0, 7.0)
        rain = np.random.exponential(1.0)
        wind = np.random.normal(24.0, 6.0)
    elif "healthy" in cn or "background" in cn:
        temp = np.random.normal(24.0, 4.0)
        humidity = np.random.normal(55.0, 8.0)
        rain = 0.0
        wind = np.random.normal(10.0, 3.0)
    else:
        temp = np.random.normal(28.0, 4.0)
        humidity = np.random.normal(82.0, 6.0)
        rain = np.random.exponential(2.0)
        wind = np.random.normal(15.0, 4.0)
    return [float(np.clip((temp+10)/60,0,1)), float(np.clip(humidity/100,0,1)), float(np.clip(rain/50,0,1)), float(np.clip(wind/80,0,1))]

def create_tf_dataset(df, batch_size=BATCH_SIZE, is_training=True):
    file_paths = df['file_path'].values
    class_indices = df['class_idx'].values
    class_names = df['class_name'].values
    n_cls = len(CLASS_NAMES)

    def generator():
        while True:  # Infinite generator stream
            indices = np.arange(len(file_paths))
            if is_training: np.random.shuffle(indices)
            for idx in indices:
                p = file_paths[idx]
                c_idx = class_indices[idx]
                c_name = class_names[idx]
                img_bgr = cv2.imread(p)
                if img_bgr is None:
                    img_rgb = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
                else:
                    img_rgb = cv2.cvtColor(cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB).astype(np.float32)
                    # MobileNetV3 Exact Preprocessing [-1, 1]
                    img_rgb = tf.keras.applications.mobilenet_v3.preprocess_input(img_rgb)
                w_vec = synthesize_weather_vector(c_name)
                one_hot = np.zeros(n_cls, dtype=np.float32)
                one_hot[c_idx] = 1.0
                yield (img_rgb, np.array(w_vec, dtype=np.float32)), one_hot

    output_sig = (
        (tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32), tf.TensorSpec(shape=(WEATHER_DIM,), dtype=tf.float32)),
        tf.TensorSpec(shape=(n_cls,), dtype=tf.float32)
    )
    ds = tf.data.Dataset.from_generator(generator, output_signature=output_sig)
    if is_training: ds = ds.shuffle(buffer_size=512)
    return ds.repeat().batch(batch_size).prefetch(tf.data.AUTOTUNE)

✅ Discovered 62669 Total Images across PlantVillage + PlantDoc.


## 🏗️ 2. Build MobileNetV3 Multimodal Fusion Network with Data Augmentation

In [3]:
def build_mobilenet_multimodal(num_classes=NUM_CLASSES):
    img_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="leaf_image_input")
    weather_input = layers.Input(shape=(WEATHER_DIM,), name="weather_input")

    # Data Augmentation Layer (Active only during training)
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ], name="data_augmentation")(img_input)

    base_vision = tf.keras.applications.MobileNetV3Small(
        input_tensor=aug,
        include_top=False,
        weights="imagenet"
    )
    x = layers.GlobalAveragePooling2D(name="vision_gap")(base_vision.output)
    x = layers.Dense(256, activation="relu", name="vision_dense_1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    w = layers.Dense(32, activation="relu", name="weather_dense_1")(weather_input)
    w = layers.Dense(16, activation="relu", name="weather_dense_2")(w)

    fused = layers.concatenate([x, w], name="multimodal_fusion")
    fused = layers.Dense(128, activation="relu", name="fusion_dense")(fused)
    fused = layers.Dropout(0.35, name="fusion_dropout")(fused)

    output = layers.Dense(num_classes, activation="softmax", name="disease_prediction")(fused)
    model = Model(inputs=[img_input, weather_input], outputs=output, name="AgroIntelli_MobileNetV3_Multimodal")
    return model, base_vision

model_mobilenet, base_vision = build_mobilenet_multimodal()
model_mobilenet.summary()

Model: "AgroIntelli_MobileNetV3_Multimodal"

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)     ┃ Output Shape   ┃  Param # ┃ Connected to    ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ leaf_image_input │ (None, 224,    │        0 │ -               │
│ (InputLayer)     │ 224, 3)        │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ data_augmentati… │ (None, 224,    │        0 │ leaf_image_inp… │
│ (Sequential)     │ 224, 3)        │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ rescaling        │ (None, 224,    │        0 │ data_augmentat… │
│ (Rescaling)      │ 224, 3)        │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ conv (Conv2D)    │ (None, 112,    │      432 │ rescaling[0][0] │
│                  │ 112, 16)       │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ conv_bn          │ (None, 112,    │       64 │ conv[0][0]      │
│ (BatchNormaliza… │ 112, 16)       │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ activation       │ (None, 112,    │        0 │ conv_bn[0][0]   │
│ (Activation)     │ 112, 16)       │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_d… │ (None, 113,    │        0 │ activation[0][… │
│ (ZeroPadding2D)  │ 113, 16)       │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_d… │ (None, 56, 56, │      144 │ expanded_conv_… │
│ (DepthwiseConv2… │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_d… │ (None, 56, 56, │       64 │ expanded_conv_… │
│ (BatchNormaliza… │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ re_lu (ReLU)     │ (None, 56, 56, │        0 │ expanded_conv_… │
│                  │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_s… │ (None, 1, 1,   │        0 │ re_lu[0][0]     │
│ (GlobalAverageP… │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_s… │ (None, 1, 1,   │      136 │ expanded_conv_… │
│ (Conv2D)         │ 8)             │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_s… │ (None, 1, 1,   │        0 │ expanded_conv_… │
│ (ReLU)           │ 8)             │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_s… │ (None, 1, 1,   │      144 │ expanded_conv_… │
│ (Conv2D)         │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ add (Add)        │ (None, 1, 1,   │        0 │ expanded_conv_… │
│                  │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ re_lu_1 (ReLU)   │ (None, 1, 1,   │        0 │ add[0][0]       │
│                  │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ multiply         │ (None, 1, 1,   │        0 │ re_lu_1[0][0]   │
│ (Multiply)       │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_s… │ (None, 56, 56, │        0 │ re_lu[0][0],    │
│ (Multiply)       │ 16)            │          │ multiply[0][0]  │
├──────────────────┼────────────────┼──────────┼─────────────────┤
│ expanded_conv_p… │ (None, 56, 56, │      256 │ expanded_conv_… │
│ (Conv2D)         │ 16)            │          │                 │
├──────────────────┼────────────────┼──────────

 Total params: 1,128,519 (4.30 MB)

 Trainable params: 1,115,895 (4.26 MB)

 Non-trainable params: 12,624 (49.31 KB)

## 🏋️ 3. Failsafe Safeguard & High-Accuracy Two-Stage Training Harness

In [4]:
MODELS_DIR = Path("../backend/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
primary_path = MODELS_DIR / "agrointelli_phase1_final.keras"
backup_path = MODELS_DIR / "agrointelli_phase1_final.backup.keras"

if primary_path.exists():
    try:
        shutil.copyfile(str(primary_path), str(backup_path))
        print(f"🛡️ Safeguard: Backup model saved -> {backup_path.name}")
    except Exception as e: print(f"⚠️ Backup notice: {e}")

if len(df_dataset) > 0:
    train_df, val_df = train_test_split(df_dataset, test_size=0.15, stratify=df_dataset['class_name'], random_state=SEED)
    train_ds = create_tf_dataset(train_df, batch_size=BATCH_SIZE, is_training=True)
    val_ds = create_tf_dataset(val_df, batch_size=BATCH_SIZE, is_training=False)
    
    steps_per_epoch = len(train_df) // BATCH_SIZE
    val_steps = len(val_df) // BATCH_SIZE
    print(f"📊 Training steps per epoch: {steps_per_epoch} | Validation steps: {val_steps}")

    loss_func = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    # STAGE 1: Frozen Backbone Training (Classification Head)
    print("\n⚡ STAGE 1: Training Classification Head (Backbone Frozen, 5 Epochs)...")
    base_vision.trainable = False
    model_mobilenet.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=loss_func,
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy")]
    )
    history_s1 = model_mobilenet.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        steps_per_epoch=steps_per_epoch,
        validation_steps=val_steps,
        verbose=1
    )
    gc.collect()

    # STAGE 2: Fine-Tuning Top Layers
    print("\n🔥 STAGE 2: Fine-Tuning Top Backbone Layers (15 Epochs max)...")
    base_vision.trainable = True
    for layer in base_vision.layers[:-35]: layer.trainable = False
    
    model_mobilenet.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=loss_func,
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy")]
    )
    
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(filepath=str(primary_path), save_best_only=True, monitor="val_accuracy", mode="max", verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    ]
    
    try:
        history_s2 = model_mobilenet.fit(
            train_ds,
            validation_data=val_ds,
            epochs=15,
            steps_per_epoch=steps_per_epoch,
            validation_steps=val_steps,
            callbacks=callbacks,
            verbose=1
        )
        print(f"\n🎉 Training Complete! High-accuracy model saved to {primary_path.name}")
    except Exception as e:
        print(f"\n⚠️ Training interrupted: {e}")
        if backup_path.exists():
            shutil.copyfile(str(backup_path), str(primary_path))
            print(f"🛡️ Failsafe Triggered: Restored stable backup from {backup_path.name}")
    finally:
        gc.collect()

🛡️ Safeguard: Backup model saved -> agrointelli_phase1_final.backup.keras
📊 Training steps per epoch: 3329 | Validation steps: 587

⚡ STAGE 1: Training Classification Head (Backbone Frozen, 5 Epochs)...
Epoch 1/5
3329/3329 ━━━━━━━━━━━━━━━━━━━━ 324s 95ms/step - accuracy: 0.8058 - loss: 1.3627 - top3_accuracy: 0.9273 - val_accuracy: 0.9139 - val_loss: 1.0190 - val_top3_accuracy: 0.9845
Epoch 2/5
3329/3329 ━━━━━━━━━━━━━━━━━━━━ 294s 88ms/step - accuracy: 0.8937 - loss: 1.1179 - top3_accuracy: 0.9739 - val_accuracy: 0.9302 - val_loss: 0.9744 - val_top3_accuracy: 0.9865
Epoch 3/5
3329/3329 ━━━━━━━━━━━━━━━━━━━━ 296s 89ms/step - accuracy: 0.9089 - loss: 1.0673 - top3_accuracy: 0.9791 - val_accuracy: 0.9366 - val_loss: 0.9345 - val_top3_accuracy: 0.9885
Epoch 4/5
3329/3329 ━━━━━━━━━━━━━━━━━━━━ 292s 88ms/step - accuracy: 0.9162 - loss: 1.0420 - top3_accuracy: 0.9826 - val_accuracy: 0.9484 - val_loss: 0.9047 - val_top3_accuracy: 0.9899
Epoch 5/5
3329/3329 ━━━━━━━━━━━━━━━━━━━━ 298s 89ms/step - acc

## 💾 4. Export Model Weights to Backend

In [5]:
mobilenet_export = MODELS_DIR / "model_mobilenet.keras"
model_mobilenet.save(str(mobilenet_export))
print(f"✅ MobileNetV3 model saved to: {mobilenet_export.resolve()}")
print("🚀 Notebook 02 complete!")

✅ MobileNetV3 model saved to: C:\Users\rahul\Desktop\pd\AgroIntelli_Project\backend\models\model_mobilenet.keras
🚀 Notebook 02 complete!
